In [11]:
import mlflow
from pathlib import Path

from src.build_spark import spark_session_context

from src.inference import predict_rul
from src.tracking import fetch_run_id
from src.data_processing.data_loader import load_cmapss_raw

In [12]:
project_root = Path("/home/aanchal/nasa_c_mapss")

mlflow.set_tracking_uri("file://" + str(project_root / "mlruns"))

In [13]:
print(Path.cwd())
print(mlflow.get_tracking_uri())

/home/aanchal/nasa_c_mapss/code
file:///home/aanchal/nasa_c_mapss/mlruns


In [35]:
subset_id = "FD004"
unit_id = "5"

In [36]:
with spark_session_context(app_name="cmapss-rul-estimation") as spark:

    test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData/test_FD004.txt")    

    raw_trajectory = (test_data.where(f"unit_id = {unit_id}").orderBy("cycle"))    
    raw_trajectory.show()
    
    observations = [row.asDict(recursive=True) for row in raw_trajectory.collect()]

+-------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|unit_id|cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|
+-------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|      5|    1|  42.0003|     0.84|    100.0|   445.0|  549.73| 1352.04| 1123.66|    3.91|    5.71|  138.31| 2211.86| 8323.79|     1.02|     42.3|   130.54|  2387.91|  8082.97|   9.3695|     0.02|    331.0|   2212.0|    100.0|    10.59|   6.3617|
|      5|   

In [37]:
model_types = ("RandomForestRegressor", "XGBRegressor", "LSTMRegressor")

predictions = {
    model_type: predict_rul(
        subset_id=subset_id,
        training_run_id=fetch_run_id(
            subset_id, experiment_type="training", model_type=model_type
        ),
        observations=observations,
    )
    for model_type in model_types
}

predictions

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished


{'RandomForestRegressor': 88.44,
 'XGBRegressor': 105.56930541992188,
 'LSTMRegressor': 89.98958587646484}

# Server Check

In [17]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

In [18]:
BASE_URL = "http://localhost:8000"

In [19]:
def call_api(path, payload=None):
    body = json.dumps(payload).encode() if payload is not None else None
    request = Request(
        f"{BASE_URL}{path}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST" if payload is not None else "GET",
    )

    with urlopen(request) as response:
        return json.load(response)

In [20]:
print(call_api("/health"))

{'status': 'ok'}


In [21]:
# Single Cycle Observation

observations = [{"unit_id": 1,
                 "cycle": 127,
                 "setting_1": 0.43,
                 "setting_2": 0.0,
                 "setting_3": 100.0,
                 **{f"sensor_{number}": 2.0 for number in range(1, 22)}}]

In [22]:
predictions = {
    model_type: call_api("/predict", {
        "subset_id": "FD004",
        "model_type": model_type,
        "observations": observations,
    })
    for model_type in model_types
}

predictions

{'subset_id': 'FD004',
 'model_type': 'RandomForestRegressor',
 'predicted_rul': 99.93}

In [23]:
# with spark_session_context(app_name="cmapss-api-test") as spark:
#     test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData/test_FD004.txt")
#     raw_trajectory = (test_data.where(f"unit_id = {unit_id}").orderBy("cycle"))    
#     observations = [raw_trajectory.orderBy("cycle", ascending=False).limit(1).collect()[0].asDict()]

In [24]:
# # Multiple Cycle Observations

# with spark_session_context(app_name="cmapss-api-test") as spark:
#     test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData/test_FD004.txt")
#     raw_trajectory = (test_data.where(f"unit_id = {unit_id}").orderBy("cycle"))    
#     observations = [row.asDict() for row in raw_trajectory.collect()]

# Running Tests

In [25]:
from pyspark.sql import Window, functions as F

In [31]:
configured_subsets = tuple(("FD001", "FD002", "FD003", "FD004"))

request_count = 50
random_seed = 42

In [38]:
base_count, remainder = divmod(request_count, len(configured_subsets))
test_cases = []

with spark_session_context(app_name="cmapss-prometheus-data") as spark:
    for subset_index, current_subset in enumerate(configured_subsets):
        subset_count = base_count + (subset_index < remainder)
        
        test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData" / f"test_{current_subset}.txt")
        
        random_cycle = Window.partitionBy("unit_id").orderBy(F.rand(random_seed + subset_index))
        
        sampled_rows = (test_data.withColumn("cycle_rank", F.row_number().over(random_cycle))
                        .where(F.col("cycle_rank") == 1)
                        .drop("cycle_rank")
                        .orderBy(F.rand(random_seed + 1000 + subset_index))
                        .limit(subset_count)
                        .collect())
        
        if len(sampled_rows) != subset_count:
            raise ValueError(f"{current_subset} has fewer than {subset_count} test engines")
        
        test_cases.extend((current_subset, row.asDict(recursive=True)) for row in sampled_rows)

In [39]:
successful_predictions = []
failed_predictions = []

for model_type in model_types:
    for current_subset, observation in test_cases:
        payload = {"subset_id": current_subset,
                   "model_type": model_type,
                   "observations": [observation]}
        
        try:
            successful_predictions.append(call_api("/predict", payload))
        
        except (HTTPError, URLError) as error:
            failed_predictions.append({"subset_id": current_subset,
                                       "model_type": model_type,
                                       "unit_id": observation["unit_id"],
                                       "error": str(error)})

In [40]:
failed_predictions[:5]

[{'subset_id': 'FD001',
  'model_type': 'LSTMRegressor',
  'unit_id': 22,
  'error': 'HTTP Error 500: Internal Server Error'},
 {'subset_id': 'FD001',
  'model_type': 'LSTMRegressor',
  'unit_id': 81,
  'error': 'HTTP Error 500: Internal Server Error'},
 {'subset_id': 'FD001',
  'model_type': 'LSTMRegressor',
  'unit_id': 70,
  'error': 'HTTP Error 500: Internal Server Error'},
 {'subset_id': 'FD001',
  'model_type': 'LSTMRegressor',
  'unit_id': 16,
  'error': 'HTTP Error 500: Internal Server Error'},
 {'subset_id': 'FD001',
  'model_type': 'LSTMRegressor',
  'unit_id': 99,
  'error': 'HTTP Error 500: Internal Server Error'}]

In [41]:
print({"requested": len(test_cases) * len(model_types),
       "successful": len(successful_predictions),
       "failed": len(failed_predictions),
       "subsets": configured_subsets})

{'requested': 150, 'successful': 100, 'failed': 50, 'subsets': ('FD001', 'FD002', 'FD003', 'FD004')}
